# TDT4173 – Submission 2

Group 71  
Students: Marco Josefsen, Theresa Kieu-Mi Tran, Bilal Saher
Kaggle team name: [71] Three Blind Mice

Score: 0.99172

In [ ]:
# -----------------------------
# Config
# -----------------------------
from pathlib import Path

RAW_DIR = Path(".")
OUTPUT_DIR = RAW_DIR
OUT_NAME = "submission_2.csv"
DEVICE = "cuda"
SEEDS = [1337, 2024, 7]
N_THREADS = 4
UNIT_ROUNDS, UNIT_DEPTH = 300, 6
POOLED_ROUNDS, POOLED_DEPTH = 890, 8

BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    learning_rate=0.05,
    min_child_weight=20,
    subsample=0.7,
    colsample_bytree=0.8,
    reg_lambda=20.0,
    reg_alpha=0.5,
    gamma=0.1,
    max_bin=256,
)


In [ ]:
# -----------------------------
# Imports and input files
# -----------------------------
import gc
import io
import json
import os
import re
import subprocess
import time
import warnings

import numpy as np
import pandas as pd
import scipy
import xgboost as xgb
import yaml
from IPython.display import display
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=FutureWarning)
if xgb.__version__ != "3.4.1":
    raise RuntimeError("Install XGBoost 3.4.1 before running this notebook.")
if DEVICE != "cpu" and not DEVICE.startswith("cuda"):
    raise ValueError("DEVICE must be 'cpu' or 'cuda'.")

RAW_DIR = RAW_DIR.expanduser().resolve()
OUTPUT_DIR = OUTPUT_DIR.expanduser().resolve()
SP = str(RAW_DIR)
WV_FILE = RAW_DIR / "data/kernel/Synthetic_water_value_2015_2024.csv"
REQUIRED_FILES = [
    "sample_submission.csv",
    "prediction_mapping.csv",
    "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "data/kernel/Historical_inflow_1958_2025.csv",
    "data/kernel/Historical_volume_2015_2024.csv",
    "data/kernel/Synthetic_water_value_2015_2024.csv",
    "data/kernel/Unit_commitment_decisions.csv",
    "data/extended/Constraint_min_flow.csv",
    "data/extended/Constraint_min_volume.csv",
    "data/extended/Tokke_Vinje_topology.yaml",
]
missing = [name for name in REQUIRED_FILES if not (RAW_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Check RAW_DIR. Missing input files: {missing}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("Data:", RAW_DIR)
print("XGBoost:", xgb.__version__, "| NumPy:", np.__version__, "| pandas:", pd.__version__)


In [ ]:
# -----------------------------
# Preprocessing helpers
# -----------------------------
import io, re

H = 168
TZ_LOCAL = "Europe/Oslo"
USE_END_OF_WEEK_VOLUME = False


def read_ts(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df.set_index("date").sort_index()


def to_hourly_grid(df, start, end):
    hourly = df.resample("h").mean()
    grid = pd.date_range(start, end, freq="h", tz="UTC")
    return hourly.reindex(grid).interpolate("time", limit_direction="both")


def drop_constant(df):
    return df.drop(columns=[c for c in df.columns if df[c].nunique() <= 1])


def dedupe_columns(df):
    keep = []
    for c in df.columns:
        if next((k for k in keep if df[k].equals(df[c])), None) is None:
            keep.append(c)
    return df[keep]


def windows(series_or_df, starts, lookback=0):
    """(n_cases, H + lookback, n_cols) hourly values for each case window [start − lookback, start + H)."""
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df
    pos = df.index.get_indexer(starts)
    if not ((pos >= lookback).all() and (pos + H - 1 < len(df)).all()):
        raise ValueError("Case window outside the hourly input grid.")
    idx = pos[:, None] + np.arange(-lookback, H)[None, :]
    return df.to_numpy(dtype=np.float32)[idx]


def _csv_roundtrip(df, float_format):
    buf = io.StringIO()
    df.to_csv(buf, index=False, float_format=float_format)
    buf.seek(0)
    return pd.read_csv(buf)


In [ ]:
# -----------------------------
# Load and preprocess the raw data
# -----------------------------
def preprocess(SP):
    KERNEL, EXTENDED = f"{SP}/data/kernel", f"{SP}/data/extended"
    price_raw = read_ts(f"{KERNEL}/Historical_day_ahead_price_2015_2025.csv")
    inflow_raw = read_ts(f"{KERNEL}/Historical_inflow_1958_2025.csv")
    volume_raw = read_ts(f"{KERNEL}/Historical_volume_2015_2024.csv")
    wv_raw = read_ts(f"{KERNEL}/Synthetic_water_value_2015_2024.csv")
    minvol_raw = read_ts(f"{EXTENDED}/Constraint_min_volume.csv")
    minflow_raw = read_ts(f"{EXTENDED}/Constraint_min_flow.csv")
    uc = pd.read_csv(f"{KERNEL}/Unit_commitment_decisions.csv")
    test_map = pd.read_csv(f"{SP}/prediction_mapping.csv")
    sample_sub = pd.read_csv(f"{SP}/sample_submission.csv")
    topo = yaml.safe_load(open(f"{EXTENDED}/Tokke_Vinje_topology.yaml", encoding="utf-8"))

    # Put the series on a complete hourly UTC grid.
    GRID_START, GRID_END = "2014-12-01", "2025-01-08 23:00"
    price = to_hourly_grid(price_raw, GRID_START, GRID_END)["Day-ahead price (EUR/MWh)"].rename(
        "price"
    )
    inflow = drop_constant(to_hourly_grid(inflow_raw, GRID_START, GRID_END))
    minvol = drop_constant(to_hourly_grid(minvol_raw, "2015-01-01", GRID_END))
    minflow = drop_constant(to_hourly_grid(minflow_raw, "2015-01-01", GRID_END))
    volume = drop_constant(volume_raw)
    wv = dedupe_columns(wv_raw)

    # Map plants to their connected reservoirs.
    model = topo["model"]
    max_vol = {r: v["max_vol"] for r, v in model["reservoir"].items()}
    conns = pd.DataFrame(topo["connections"])

    def follow_to_reservoir(node, direction):
        key, other, other_type = (
            ("to", "from", "from_type") if direction == "up" else ("from", "to", "to_type")
        )
        frontier, seen, found = [node], set(), []
        while frontier:
            n = frontier.pop()
            for _, c in conns[conns[key] == n].iterrows():
                if c[other_type] == "reservoir":
                    found.append(c[other])
                elif c[other_type] in ("tunnel", "river") and c[other] not in seen:
                    seen.add(c[other])
                    frontier.append(c[other])
        return found

    plants = list(model["plant"].keys())
    plant_map = {
        p: {"up": follow_to_reservoir(p, "up"), "down": follow_to_reservoir(p, "down")}
        for p in plants
    }
    target_cols = [c for c in sample_sub.columns if c != "Run No"]
    UNITS = list(
        dict.fromkeys(re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in target_cols)
    )
    unit_plant = {u: u.rsplit("_", 1)[0] for u in UNITS}

    def plant_wv(day_index):
        out = {}
        for p, m in plant_map.items():
            out[f"wv_up_{p}"] = wv_raw.loc[day_index, m["up"]].mean(axis=1).to_numpy()
            out[f"wv_down_{p}"] = (
                wv_raw.loc[day_index, m["down"]].mean(axis=1).to_numpy()
                if m["down"]
                else np.zeros(len(day_index))
            )
        return out

    cases = pd.concat(
        [
            uc[["Run No", "starttime"]].assign(split="train"),
            test_map[["Run No", "starttime"]].assign(split="test"),
        ],
        ignore_index=True,
    )
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
    cases = cases.sort_values("Run No").reset_index(drop=True)
    if not cases["Run No"].is_unique or cases["starttime"].isna().any():
        raise ValueError("Raw cases need unique identifiers and valid start times.")
    Y_train = (
        uc.set_index("Run No")[target_cols]
        .to_numpy(dtype=np.float32)
        .reshape(len(uc), len(UNITS), H)
    )

    starts = pd.DatetimeIndex(cases["starttime"])
    N = len(cases)
    P = windows(price, starts)[..., 0]
    P_prev = windows(price, starts, lookback=H)[:, :H, 0]
    INF = windows(inflow, starts)
    MINV = windows(minvol, starts) if minvol.shape[1] else np.zeros((N, H, 0), np.float32)
    MINF = windows(minflow, starts) if minflow.shape[1] else np.zeros((N, H, 0), np.float32)

    # Case-level features.
    day0 = starts.normalize()
    day_end = day0 + pd.Timedelta("7D")
    f = {"Run No": cases["Run No"].to_numpy()}
    f["year"] = starts.year
    f["month"] = starts.month
    doy = starts.dayofyear.to_numpy()
    f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    f["price_mean"], f["price_std"] = P.mean(1), P.std(1)
    f["price_min"], f["price_max"] = P.min(1), P.max(1)
    f["price_q10"], f["price_q90"] = np.quantile(P, 0.1, axis=1), np.quantile(P, 0.9, axis=1)
    f["price_prev_week_mean"] = P_prev.mean(1)
    f["price_week_over_week"] = f["price_mean"] - f["price_prev_week_mean"]
    f["price_daily_amplitude"] = (P.reshape(N, 7, 24).max(2) - P.reshape(N, 7, 24).min(2)).mean(1)
    for r in volume.columns:
        v0 = volume.loc[day0, r].to_numpy()
        f[f"vol_{r}"] = v0
        f[f"fill_{r}"] = v0 / max_vol[r]
        if USE_END_OF_WEEK_VOLUME:
            f[f"dvol_{r}"] = volume.loc[day_end, r].to_numpy() - v0
    f["fill_total"] = sum(volume.loc[day0, r].to_numpy() for r in volume.columns) / sum(
        max_vol[r] for r in volume.columns
    )
    for r in wv.columns:
        f[f"wv0_{r}"] = wv.loc[day0, r].to_numpy()
        f[f"wv7_{r}"] = wv.loc[day_end, r].to_numpy()
        f[f"dwv_{r}"] = f[f"wv7_{r}"] - f[f"wv0_{r}"]
    pw_end = plant_wv(day_end)
    for p in plants:
        up, down = pw_end[f"wv_up_{p}"], pw_end[f"wv_down_{p}"]
        f[f"wvup_{p}"], f[f"wvnet_{p}"] = up, up - down
        f[f"spread_{p}"] = f["price_mean"] - up
        f[f"frac_price_gt_wv_{p}"] = (P > up[:, None]).mean(1)
    clim_src = inflow_raw.loc[:"2014-12-31", inflow.columns]
    clim = clim_src.groupby(clim_src.index.dayofyear).mean()
    inf_week = INF.mean(1)
    clim_days = ((doy[:, None] - 1 + np.arange(7)[None, :]) % 365) + 1
    clim_week = clim.to_numpy()[clim_days - 1].mean(1)
    for j, r in enumerate(inflow.columns):
        f[f"inflow_{r}"] = inf_week[:, j]
    f["inflow_total"] = inf_week.sum(1)
    f["inflow_total_anomaly"] = inf_week.sum(1) / (clim_week.sum(1) + 1e-6)
    for j, r in enumerate(minflow.columns):
        f[f"minflow_{r}"] = MINF[:, :, j].mean(1)
    for j, r in enumerate(minvol.columns):
        f[f"minvol_{r}"] = MINV[:, :, j].mean(1)
    case_feat = pd.DataFrame(f)
    case_feat.insert(1, "starttime", cases["starttime"].to_numpy())
    case_feat.insert(2, "split", cases["split"].to_numpy())
    case_feat["fold"] = np.where(case_feat["split"] == "train", case_feat["year"], -1)

    # Hourly features.
    t = np.arange(H)
    ts = (
        starts.tz_convert(None).to_numpy(dtype="datetime64[ns]")[:, None]
        + (t * np.timedelta64(1, "h"))[None, :]
    )
    ts_local = pd.DatetimeIndex(ts.ravel()).tz_localize("UTC").tz_convert(TZ_LOCAL)
    hf = {
        "Run No": np.repeat(cases["Run No"].to_numpy(), H),
        "t": np.tile(t, N),
        "day_in_horizon": np.tile(t // 24, N),
        "hour_local": ts_local.hour.to_numpy(),
        "dow_local": ts_local.dayofweek.to_numpy(),
        "is_weekend": (ts_local.dayofweek >= 5).astype(np.int8),
    }
    hs = {}
    hs["price"] = P
    hs["price_minus_mean"] = P - P.mean(1, keepdims=True)
    hs["price_z"] = hs["price_minus_mean"] / (P.std(1, keepdims=True) + 1e-6)

    hs["price_rank_week"] = P.argsort(1, kind="quicksort").argsort(1, kind="quicksort") / (H - 1)
    Pd = P.reshape(N, 7, 24)
    hs["price_rank_day"] = (
        Pd.argsort(2, kind="quicksort").argsort(2, kind="quicksort") / 23
    ).reshape(N, H)
    hs["price_minus_daymean"] = (Pd - Pd.mean(2, keepdims=True)).reshape(N, H)
    hs["price_diff_prev"] = np.diff(P, axis=1, prepend=P_prev[:, -1:])
    hs["price_diff_next"] = np.concatenate(
        [P[:, 1:] - P[:, :-1], np.zeros((N, 1), np.float32)], axis=1
    )
    k = 6
    pad = np.pad(P, ((0, 0), (k, k)), mode="edge")
    hs["price_roll13"] = np.stack([pad[:, i : i + H] for i in range(2 * k + 1)]).mean(0)
    for p in plants:
        hs[f"price_minus_wvup_{p}"] = P - pw_end[f"wv_up_{p}"][:, None]
    hs["inflow_total_h"] = INF.sum(2)
    for j, r in enumerate(minflow.columns):
        hs[f"minflow_h_{r}"] = MINF[:, :, j]
    for j, r in enumerate(minvol.columns):
        hs[f"minvol_h_{r}"] = MINV[:, :, j]
    for name, arr in hs.items():
        hf[name] = arr.astype(np.float32).ravel()
    hourly = pd.DataFrame(hf)
    Y_all = np.full((N, len(UNITS), H), np.nan, dtype=np.float32)
    train_mask = (cases["split"] == "train").to_numpy()
    if not np.array_equal(cases.loc[train_mask, "Run No"].to_numpy(), uc["Run No"].to_numpy()):
        raise ValueError("Label rows do not match the sorted training case IDs.")
    Y_all[train_mask] = Y_train
    for u_i, u in enumerate(UNITS):
        hourly[f"y_{u}"] = Y_all[:, u_i, :].ravel()

    hourly_feature_cols = list(hs.keys()) + [
        "t",
        "day_in_horizon",
        "hour_local",
        "dow_local",
        "is_weekend",
    ]
    static_feature_cols = [
        c for c in case_feat.columns if c not in ("Run No", "starttime", "split", "fold")
    ]
    y_cols = [f"y_{u}" for u in UNITS]
    meta = {
        "units": UNITS,
        "unit_plant": unit_plant,
        "plant_map": plant_map,
        "submission_columns": sample_sub.columns.tolist(),
        "hourly_feature_cols": hourly_feature_cols,
        "static_feature_cols": static_feature_cols,
        "y_cols": y_cols,
    }

    # Retain the precision used in the original preprocessing CSVs.
    hourly_cols = ["Run No"] + hourly_feature_cols
    is_train_row = np.repeat(train_mask, H)
    cases_s1 = _csv_roundtrip(case_feat, "%.6g")
    h_tr = _csv_roundtrip(
        hourly.loc[is_train_row, hourly_cols + y_cols].astype({c: "int8" for c in y_cols}), "%.6g"
    )
    h_te = _csv_roundtrip(hourly.loc[~is_train_row, hourly_cols], "%.6g")
    return cases_s1, h_tr, h_te, meta


In [ ]:
# -----------------------------
# Scheduling features
# -----------------------------
def window_mean(a, width, forward):
    a = np.asarray(a, dtype=np.float64)
    h = a.shape[1]
    t = np.arange(h)
    left = t if forward else np.maximum(0, t - width + 1)
    right = np.minimum(h, t + width) if forward else t + 1
    cumulative = np.pad(np.cumsum(a, axis=1), ((0, 0), (1, 0)))
    return (cumulative[:, right] - cumulative[:, left]) / (right - left)


def remaining_sum(a):
    return np.cumsum(np.asarray(a, dtype=np.float64)[:, ::-1], axis=1)[:, ::-1]


def block_total(values, active):
    left = np.zeros_like(values, dtype=float)
    right = np.zeros_like(left)
    for t in range(values.shape[1]):
        previous = left[:, t - 1] if t else 0
        left[:, t] = np.where(active[:, t], previous + values[:, t], 0)
    for t in range(values.shape[1] - 1, -1, -1):
        following = right[:, t + 1] if t + 1 < values.shape[1] else 0
        right[:, t] = np.where(active[:, t], following + values[:, t], 0)
    return left + right - np.where(active, values, 0)


def engineer_cases(source_cases, metadata):
    result = source_cases.drop(columns=[c for c in source_cases if c.startswith("dvol_")]).copy()
    new = {}
    for plant, mapping in metadata["plant_map"].items():
        side_fill = {}
        for side in ("up", "down"):
            columns = [f"fill_{r}" for r in mapping[side]]
            if columns and all(c in result for c in columns):
                side_fill[side] = result[columns].mean(axis=1).to_numpy()
                new[f"fe_initial_fill_{side}_{plant}"] = side_fill[side]
        if "up" in side_fill and "down" in side_fill:
            new[f"fe_initial_fill_up_minus_down_{plant}"] = side_fill["up"] - side_fill["down"]
    for column in source_cases:
        if not column.startswith("vol_"):
            continue
        reservoir = column[len("vol_") :]
        inflow_column = f"inflow_{reservoir}"
        if inflow_column not in result:
            continue
        initial = result[column].to_numpy(dtype=float)
        incoming = result[inflow_column].to_numpy(dtype=float) * H * 3600 / 1e6
        denominator = initial + incoming
        new[f"fe_inflow_share_of_initial_plus_week_{reservoir}"] = np.divide(
            incoming, denominator, out=np.zeros_like(incoming), where=denominator > 0
        )
    additions = pd.DataFrame(new, index=result.index, dtype=np.float32)
    return pd.concat([result, additions], axis=1), list(additions.columns)


def engineer_hourly(source_hourly, source_cases, metadata):
    result = source_hourly.drop(columns=[c for c in source_hourly if c.startswith("dvol_")])
    result = result.sort_values(["Run No", "t"]).reset_index(drop=True)
    ids = result["Run No"].drop_duplicates().to_numpy()
    if not (
        result.groupby("Run No").size().eq(H).all()
        and np.array_equal(result["t"].to_numpy(), np.tile(np.arange(H), len(ids)))
    ):
        raise ValueError("Each case needs exactly hours 0 through 167.")
    aligned = source_cases.set_index("Run No").loc[ids]
    shape = (len(ids), H)
    price = result["price"].to_numpy(dtype=float).reshape(shape)
    count_remaining = H - np.arange(H)
    new = {}
    for width in (6, 24):
        for direction, forward in (("past", False), ("next", True)):
            new[f"fe_price_minus_{direction}{width}h_mean"] = price - window_mean(
                price, width, forward
            )
    new["fe_remaining_price_max_minus_current"] = (
        np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price
    )
    new["fe_remaining_price_mean_minus_current"] = remaining_sum(price) / count_remaining - price
    extra = {}
    for plant in metadata["plant_map"]:
        spread = price - aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        above = spread > 0
        new[f"fe_price_minus_wvnet_{plant}"] = (
            price - aligned[f"wvnet_{plant}"].to_numpy(dtype=float)[:, None]
        )
        new[f"fe_above_wv_block_hours_{plant}"] = block_total(np.ones_like(spread), above)
        new[f"fe_remaining_above_wv_fraction_{plant}"] = remaining_sum(above) / count_remaining
        new[f"fe_remaining_positive_spread_mean_{plant}"] = (
            remaining_sum(np.maximum(spread, 0)) / count_remaining
        )
        water_value = aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        extra[f"fe_relative_price_spread_{plant}"] = spread / (
            np.abs(price) + np.abs(water_value) + 1.0
        )
    inflow = result["inflow_total_h"].to_numpy(dtype=float).reshape(shape)
    volume = inflow * 3600 / 1e6
    new["fe_inflow_before_hour_mm3"] = np.pad(np.cumsum(volume, axis=1)[:, :-1], ((0, 0), (1, 0)))
    new["fe_inflow_remaining_mm3"] = remaining_sum(volume)
    new["fe_inflow_minus_week_mean"] = inflow - inflow.mean(axis=1, keepdims=True)
    for column in result:
        if column.startswith("minvol_h_"):
            reservoir = column[len("minvol_h_") :]
            initial = aligned[f"vol_{reservoir}"].to_numpy(dtype=float)[:, None]
            new[f"fe_initial_volume_minus_minvol_{reservoir}"] = initial - result[column].to_numpy(
                dtype=float
            ).reshape(shape)
    new.update(extra)
    additions = pd.DataFrame({name: a.ravel().astype(np.float32) for name, a in new.items()})
    return pd.concat([result, additions], axis=1), list(additions.columns)


In [ ]:
# -----------------------------
# Build the scheduling tables
# -----------------------------
def build_control_tables(SP):
    cases_s1, h_tr_s1, h_te_s1, meta = preprocess(SP)
    cases_fe, added_static = engineer_cases(cases_s1, meta)
    h_tr, added_hourly = engineer_hourly(h_tr_s1, cases_s1, meta)
    h_te, added_te = engineer_hourly(h_te_s1, cases_s1, meta)
    if added_te != added_hourly:
        raise ValueError("Train and test feature schemas differ.")
    meta = dict(meta)
    meta["hourly_feature_cols"] = meta["hourly_feature_cols"] + added_hourly
    meta["static_feature_cols"] = meta["static_feature_cols"] + added_static
    return cases_fe, h_tr, h_te, meta


t0 = time.perf_counter()
cases, h_tr, h_te, meta = build_control_tables(SP)
print(f"Scheduling inputs built in {time.perf_counter() - t0:.1f}s")


In [ ]:
# -----------------------------
# Assemble hourly model inputs
# -----------------------------
UNITS, HF, SF = meta["units"], meta["hourly_feature_cols"], meta["static_feature_cols"]
UNIT_PLANT = meta["unit_plant"]
PLANTS = list(dict.fromkeys(UNIT_PLANT[u] for u in UNITS))
YC = [f"y_{u}" for u in UNITS]
CONTROL = HF + SF
print(f"control features: {len(HF)} hourly + {len(SF)} static = {len(CONTROL)}")

cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
cases = cases.sort_values("Run No").reset_index(drop=True)
h_te = h_te.reindex(columns=h_tr.columns)
hourly = (
    pd.concat([h_tr, h_te], ignore_index=True).sort_values(["Run No", "t"]).reset_index(drop=True)
)
del h_tr, h_te

N = len(cases)
if len(hourly) != N * H:
    raise ValueError("The hourly row count does not match the number of cases.")
rg = hourly["Run No"].to_numpy().reshape(N, H)
if not (
    (rg == cases["Run No"].to_numpy()[:, None]).all()
    and (hourly["t"].to_numpy().reshape(N, H) == np.arange(H)).all()
):
    raise ValueError("Case/hour alignment is inconsistent.")
LAB = (cases["split"] == "train").to_numpy()
print(f"cases: {N} ({LAB.sum()} labelled, {(~LAB).sum()} test) | hourly rows: {len(hourly):,}")
print(
    "labelled starts",
    cases.starttime[LAB].min().date(),
    "→",
    cases.starttime[LAB].max().date(),
    "| test starts",
    cases.starttime[~LAB].min().date(),
    "→",
    cases.starttime[~LAB].max().date(),
)

RUN = hourly["Run No"].to_numpy()
TT = hourly["t"].to_numpy()
Y = hourly[YC].to_numpy(np.float32)
FEAT = pd.DataFrame(
    np.hstack(
        [hourly[HF].to_numpy(np.float32), np.repeat(cases[SF].to_numpy(np.float32), 168, axis=0)]
    ),
    columns=CONTROL,
)
PRICE = hourly["price"].to_numpy(np.float64).reshape(N, 168)
MF = {
    c: hourly[c].to_numpy(np.float64).reshape(N, 168)
    for c in ["minflow_h_r_Vest_Vassdraget", "minflow_h_b_Vinjevatn", "minflow_h_b_Byrtevatn"]
}
del hourly
if len(CONTROL) != 262 or any(c.startswith("dvol_") for c in CONTROL):
    raise ValueError("Expected 262 scheduling features without observed future-storage changes.")
if not np.isfinite(FEAT.to_numpy()).all() or not np.isin(Y[np.repeat(LAB, H)], [0, 1]).all():
    raise ValueError("Inputs must be finite and training labels binary.")


In [ ]:
# -----------------------------
# Topology and hourly inflow
# -----------------------------
topo = yaml.safe_load(open(f"{SP}/data/extended/Tokke_Vinje_topology.yaml"))
MAXVOL = {r: v["max_vol"] for r, v in topo["model"]["reservoir"].items()}
GEN = {}
for g, v in topo["model"]["generator"].items():
    curves = (
        v["turb_eff_curves"] if isinstance(v["turb_eff_curves"], list) else [v["turb_eff_curves"]]
    )
    GEN[g] = dict(
        p_min=v["p_min"],
        p_max=v["p_max"],
        sc=v["startcost_const"] + v["stopcost_const"],
        q_min=min(min(c["x"]) for c in curves),
        q_max=max(max(c["x"]) for c in curves),
    )
PLANT_QMAX = {p: sum(GEN[u]["q_max"] for u in UNITS if UNIT_PLANT[u] == p) for p in PLANTS}
print(pd.DataFrame(GEN).T)

infl = pd.read_csv(f"{SP}/data/kernel/Historical_inflow_1958_2025.csv")
infl["date"] = pd.to_datetime(infl["date"], utc=True)
infl = infl.set_index("date").sort_index()
if not infl.index.is_unique:
    raise ValueError("Raw inflow timestamps are duplicated.")
st = cases["starttime"].dt.tz_convert(None).to_numpy(dtype="datetime64[ns]")
ts = pd.DatetimeIndex((st[:, None] + np.arange(168) * np.timedelta64(1, "h")).ravel()).tz_localize(
    "UTC"
)
Q = infl.reindex(ts)
if Q.isna().any().any():
    raise ValueError("Raw inflow is missing for some case hours.")
Q = {c: Q[c].to_numpy(np.float64).reshape(N, 168) for c in Q.columns}
del infl


def vol0(res):
    if f"vol_{res}" in cases:
        return cases[f"vol_{res}"].to_numpy(np.float64)
    if res in ("Hyljelihyl", "Vatjern"):
        return np.full(N, 0.5 * MAXVOL[res])
    raise KeyError(f"Missing initial storage for {res}")


In [ ]:
# -----------------------------
# Water-balance features
# -----------------------------
POOLS = {
    "Staavatn": (["Staavatn"], [], []),
    "Kjelavatn": (["Kjelavatn"], [], ["Staavatn"]),
    "Langesae": (["Langesae"], [], []),
    "Foersvatn+Bordalsvatn": (["Foersvatn", "Bordalsvatn"], [], ["Langesae"]),
    "Hyljelihyl": (["Hyljelihyl"], [], ["Kjelavatn", "Foersvatn+Bordalsvatn"]),
    "Songavatn+Bitdalsvatn": (["Songavatn", "Bitdalsvatn"], [], []),
    "Totak+Vaamarvatn+Venemo": (
        ["Totak", "Vaamarvatn", "Venemo"],
        [],
        ["Songavatn+Bitdalsvatn", "Hyljelihyl"],
    ),
    "Langeidvatn": (["Langeidvatn"], [], []),
    "Vatjern": (["Vatjern"], [], ["Langeidvatn"]),
    "Vinjevatn": (["Vinjevatn"], [], ["Totak+Vaamarvatn+Venemo", "Vatjern"]),
    "Botnedalsvatn": (["Botnedalsvatn"], [], []),
    "Byrtevatn": (["Byrtevatn"], [], []),
    "Bandak": (["Bandak"], ["r_Vest_Vassdraget"], ["Vinjevatn", "Botnedalsvatn", "Byrtevatn"]),
}
INTAKE = {
    "Hogga": "Bandak",
    "Tokke": "Vinjevatn",
    "Lio": "Byrtevatn",
    "Byrte": "Botnedalsvatn",
    "Vinje": "Totak+Vaamarvatn+Venemo",
    "Haukeli": "Vatjern",
    "Songa": "Songavatn+Bitdalsvatn",
    "Kjela": "Foersvatn+Bordalsvatn",
    "Vesle Kjela": "Kjelavatn",
}

SPILL, POOL_V0, POOL_C = {}, {}, {}
for pool, (members, extra, ups) in POOLS.items():
    C = sum(MAXVOL[r] for r in members)
    V = sum(vol0(r) for r in members)
    inflow = sum(Q[r] for r in members + extra) * 0.0036 + sum(SPILL[u] for u in ups)
    POOL_V0[pool], POOL_C[pool] = V.copy(), C
    sp = np.zeros((N, 168))
    for t in range(168):
        V = V + inflow[:, t]
        s = np.maximum(V - C, 0.0)
        sp[:, t] = s
        V = V - s
    SPILL[pool] = sp

WATER = {}
for p in PLANTS:
    pool, qh = INTAKE[p], PLANT_QMAX[p] * 0.0036
    sp = SPILL[pool]
    rem = np.cumsum(sp[:, ::-1], axis=1)[:, ::-1]
    first = np.where((sp > 0).any(1), (sp > 0).argmax(1), 400)
    WATER[f"wb_forced_frac_{p}"] = np.repeat((sp.sum(1) / (qh * 168))[:, None], 168, 1)
    WATER[f"wb_headroom_hours_{p}"] = np.repeat(
        ((POOL_C[pool] - POOL_V0[pool]) / qh)[:, None], 168, 1
    )
    WATER[f"wb_hours_to_spill_{p}"] = first[:, None] - np.arange(168)[None, :]
    WATER[f"wb_forced_share_remaining_{p}"] = rem / qh / (168 - np.arange(168))[None, :]
for pool in ["Totak+Vaamarvatn+Venemo", "Foersvatn+Bordalsvatn", "Songavatn+Bitdalsvatn"]:
    WATER[f"wb_pool_fill_{pool}"] = np.repeat((POOL_V0[pool] / POOL_C[pool])[:, None], 168, 1)
for r in ["Vatjern", "Hyljelihyl", "Vinjevatn", "Langeidvatn"]:
    WATER[f"wb_inflow_h_{r}"] = Q[r]


In [ ]:
# -----------------------------
# Shared-river features
# -----------------------------
deficit = np.maximum(
    0,
    MF["minflow_h_r_Vest_Vassdraget"]
    - Q["r_Vest_Vassdraget"]
    - MF["minflow_h_b_Vinjevatn"]
    - MF["minflow_h_b_Byrtevatn"],
)
alt = np.minimum(cases["wv7_Venemo"], cases["wv7_Vinjevatn"]).to_numpy(np.float64)
byrt = cases["wv7_Byrtevatn"].to_numpy(np.float64)
RIVER = {
    "rv_deficit_h": deficit,
    "rv_deficit_over_lio_qmin": deficit / GEN["Lio_G1"]["q_min"],
    "rv_deficit_week_mean": np.repeat(deficit.mean(1)[:, None], 168, 1),
    "rv_alt_wv_minus_byrtevatn": np.repeat((alt - byrt)[:, None], 168, 1),
    "rv_alt_wv_over_byrtevatn": np.repeat((alt / np.maximum(byrt, 0.5))[:, None], 168, 1),
}


In [ ]:
# -----------------------------
# Final feature matrix
# -----------------------------
extras = pd.DataFrame({k: np.asarray(v, np.float32).ravel() for k, v in {**WATER, **RIVER}.items()})
FEAT = pd.concat([FEAT, extras], axis=1)
WATER_NOHK = [c for c in WATER if not any(k in c for k in ("Haukeli", "Vatjern", "Langeidvatn"))]
BASE = CONTROL + list(RIVER) + WATER_NOHK
X = FEAT[BASE].to_numpy(np.float32)
if len(BASE) != 304 or len(set(BASE)) != 304 or not np.isfinite(X).all():
    raise ValueError(
        "Expected the original 304 finite B features. Use the same scheduling dataset as B."
    )
del WATER, RIVER, Q, extras
gc.collect()
print("Per-unit features:", X.shape[1], "| pooled features:", X.shape[1] + 21)

lab_rows, test_rows = np.repeat(LAB, H), np.repeat(~LAB, H)
print("Training:", int(lab_rows.sum()), "hourly rows; test:", int(test_rows.sum()))


In [ ]:
# -----------------------------
# Device check
# -----------------------------
cp = None
if DEVICE.startswith("cuda"):
    try:
        import cupy as cp

        cp.asarray([1.0], dtype=cp.float32)
        cp.cuda.get_current_stream().synchronize()
    except Exception as error:
        cp = None
        print("CuPy unavailable; using NumPy input buffers:", str(error)[:200])


def to_dev(a):
    return cp.asarray(a) if cp is not None and DEVICE.startswith("cuda") else a


def to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)


def release_memory():
    gc.collect()
    if cp is not None:
        cp.get_default_memory_pool().free_all_blocks()


if DEVICE.startswith("cuda"):
    try:
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
            capture_output=True,
            text=True,
        )
        print(gpu.stdout or gpu.stderr)
    except FileNotFoundError:
        print("nvidia-smi unavailable; checking the actual XGBoost device next.")
    px = np.arange(120, dtype=np.float32).reshape(40, 3)
    probe = xgb.XGBClassifier(n_estimators=2, tree_method="hist", device=DEVICE).fit(
        to_dev(px), to_dev((px[:, 0] > 60).astype(np.int32))
    )
    actual = json.loads(probe.get_booster().save_config())["learner"]["generic_param"]["device"]
    if not actual.startswith("cuda"):
        raise RuntimeError(
            "XGBoost cannot see a GPU. Select a GPU runtime, or explicitly choose CPU execution in the configuration."
        )
    del probe, px
else:
    print("CPU debugging mode. Do not run the full experiment locally.")

print(
    "Raw input folder:",
    RAW_DIR,
    "| XGBoost:",
    xgb.__version__,
    "| NumPy:",
    np.__version__,
    "| pandas:",
    pd.__version__,
    "| input:",
    "CuPy " + cp.__version__ if cp is not None else "NumPy",
)


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def sigm(z):
    return 1 / (1 + np.exp(-z))


In [ ]:
# -----------------------------
# Train one model per generating unit
# -----------------------------
def train_unit_branch():
    xtr = to_dev(X[lab_rows])
    dtest = xgb.DMatrix(to_dev(X[test_rows]), nthread=N_THREADS)
    predictions = {}
    with threadpool_limits(limits=N_THREADS):
        for seed in SEEDS:
            p = np.empty((test_rows.sum(), len(UNITS)), np.float32)
            for j, unit in enumerate(UNITS):
                y = Y[lab_rows, j].astype(np.int32)
                if len(np.unique(y)) == 1:
                    p[:, j] = y[0]
                else:
                    model = xgb.XGBClassifier(
                        **BASE_PARAMS,
                        n_estimators=UNIT_ROUNDS,
                        max_depth=UNIT_DEPTH,
                        device=DEVICE,
                        random_state=seed,
                        n_jobs=N_THREADS,
                    )
                    model.fit(xtr, to_dev(y), verbose=False)
                    p[:, j] = to_np(model.get_booster().predict(dtest))
                    del model
                print(f"U6 seed {seed}: {j+1}/{len(UNITS)} {unit}", flush=True)
            predictions[seed] = p
    return predictions


t0 = time.perf_counter()
PRED_U = train_unit_branch()
release_memory()
print(f"Per-unit branch completed in {(time.perf_counter()-t0)/60:.1f} min")


In [ ]:
# -----------------------------
# Pooled model helpers
# -----------------------------
OWN_TEMPLATES = [
    "price_minus_wvup_{p}",
    "fe_price_minus_wvnet_{p}",
    "fe_above_wv_block_hours_{p}",
    "fe_remaining_above_wv_fraction_{p}",
    "fe_remaining_positive_spread_mean_{p}",
    "fe_relative_price_spread_{p}",
    "wvup_{p}",
    "wvnet_{p}",
    "spread_{p}",
    "frac_price_gt_wv_{p}",
    "fe_initial_fill_up_{p}",
    "wb_forced_frac_{p}",
    "wb_headroom_hours_{p}",
    "wb_hours_to_spill_{p}",
    "wb_forced_share_remaining_{p}",
]
UNIT_CONST = {
    u: [
        GEN[u]["p_max"],
        GEN[u]["q_max"],
        GEN[u]["sc"],
        int(u.split("_G")[-1]),
        sum(UNIT_PLANT[v] == UNIT_PLANT[u] for v in UNITS),
    ]
    for u in UNITS
}


def own_block(unit, rows):
    p = UNIT_PLANT[unit]
    out = []
    for tpl in OWN_TEMPLATES:
        c = tpl.format(p=p)
        nan_ok = p == "Haukeli" and (c.startswith("wb_") or c.startswith("fe_initial_fill_up"))
        if c in FEAT and not (p == "Haukeli" and c.startswith("wb_")):
            out.append(FEAT[c].to_numpy(np.float32)[rows])
        else:
            if not nan_ok:
                raise KeyError(f"Missing own feature {c}")
            out.append(np.full(rows.sum(), np.nan, np.float32))
    const = np.tile(np.array(UNIT_CONST[unit], np.float32), (rows.sum(), 1))
    uid = np.full((rows.sum(), 1), UNITS.index(unit), np.float32)
    return np.hstack([uid, np.column_stack(out), const])


class StackIter(xgb.DataIter):
    def __init__(self, Xbase, rows):
        self.Xb, self.rows, self.i = Xbase, rows, 0
        super().__init__(release_data=True)

    def next(self, input_data):
        if self.i == len(UNITS):
            return False
        u = UNITS[self.i]
        X = np.hstack([self.Xb, own_block(u, self.rows)])
        self.batch = to_dev(X)
        self.labels = to_dev(Y[self.rows, self.i].astype(np.float32))
        input_data(data=self.batch, label=self.labels, feature_types=self.ftypes)
        self.i += 1
        return True

    def reset(self):
        self.i = 0


POOLED_TYPES = ["q"] * len(BASE) + ["c"] + ["q"] * 20


def pooled_dmatrix(rows, ref=None):
    it = StackIter(X[rows], rows)
    it.ftypes = POOLED_TYPES
    return xgb.QuantileDMatrix(
        it, max_bin=BASE_PARAMS["max_bin"], ref=ref, enable_categorical=True, nthread=N_THREADS
    )


def pooled_predict(booster):
    xb = X[test_rows]
    p = np.empty((test_rows.sum(), len(UNITS)), np.float32)
    for j, unit in enumerate(UNITS):
        dm = xgb.DMatrix(
            to_dev(np.hstack([xb, own_block(unit, test_rows)])),
            feature_types=POOLED_TYPES,
            enable_categorical=True,
            nthread=N_THREADS,
        )
        p[:, j] = to_np(booster.predict(dm))
        del dm
    return p


def train_pooled_branch():
    dtrain = pooled_dmatrix(lab_rows)
    predictions = {}
    with threadpool_limits(limits=N_THREADS):
        for seed in SEEDS:
            params = dict(
                BASE_PARAMS,
                max_depth=POOLED_DEPTH,
                device=DEVICE,
                seed=seed,
                max_cat_to_onehot=1,
                nthread=N_THREADS,
            )
            booster = xgb.train(params, dtrain, num_boost_round=POOLED_ROUNDS)
            predictions[seed] = pooled_predict(booster)
            print(f"P8 seed {seed}: {POOLED_ROUNDS} rounds complete", flush=True)
            del booster
    return predictions


In [ ]:
# -----------------------------
# Train the pooled model
# -----------------------------
t0 = time.perf_counter()
PRED_P = train_pooled_branch()
release_memory()
print(f"Pooled branch completed in {(time.perf_counter()-t0)/60:.1f} min")


In [ ]:
# -----------------------------
# Blend predictions and save CSV
# -----------------------------
def combine_predictions(unit_predictions, pooled_predictions):
    mean_unit = np.zeros_like(unit_predictions[SEEDS[0]], dtype=np.float64)
    mean_pool_logit = np.zeros_like(pooled_predictions[SEEDS[0]], dtype=np.float64)
    for seed in SEEDS:
        mean_unit += unit_predictions[seed] / len(SEEDS)
        mean_pool_logit += logit(pooled_predictions[seed]) / len(SEEDS)
    mean_pool = sigm(mean_pool_logit)
    return sigm(0.5 * logit(mean_unit) + 0.5 * logit(mean_pool))


def format_submission(probabilities):
    runs = RUN[test_rows].reshape(-1, H)[:, 0]
    columns = [f"result_committed_{u}_t{t}" for u in UNITS for t in range(H)]
    wide = probabilities.reshape(-1, H, len(UNITS)).transpose(0, 2, 1).reshape(len(runs), -1)
    result = pd.DataFrame(wide, columns=columns)
    result.insert(0, "Run No", runs)
    sample = pd.read_csv(RAW_DIR / "sample_submission.csv")
    if set(result.columns) != set(sample.columns) or set(runs) != set(sample["Run No"]):
        raise ValueError("Prediction IDs or columns do not match the sample submission.")
    result = result.set_index("Run No").loc[sample["Run No"]].reset_index()[sample.columns]
    values = result.drop(columns="Run No").to_numpy()
    if (
        result.shape != sample.shape
        or not np.isfinite(values).all()
        or not ((values >= 0) & (values <= 1)).all()
    ):
        raise ValueError("Invalid submission shape or probabilities.")
    return result


test_probabilities = combine_predictions(PRED_U, PRED_P)
submission = format_submission(test_probabilities)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / OUT_NAME
submission.to_csv(output_path, index=False)
print("Saved:", output_path.resolve(), "| shape:", submission.shape)
print("Total runtime:", round((time.time() - T_START) / 60, 1), "minutes")
display(submission.iloc[:5, :6])
